In [ ]:
# Package installs go here 
%conda install polars, plotly

/opt/anaconda3/lib/python3.12/site-packages/conda/base/context.py:201: FutureWarning: Adding 'defaults' to channel list implicitly is deprecated and will be removed in 25.3. 

To remove this warning, please choose a default channel explicitly with conda's regular configuration system, e.g. by adding 'defaults' to the list of channels:

  conda config --add channels defaults

For more information see https://docs.conda.io/projects/conda/en/stable/user-guide/configuration/use-condarc.html

  deprecated.topic(
/opt/anaconda3/lib/python3.12/site-packages/conda/base/context.py:201: FutureWarning: Adding 'defaults' to channel list implicitly is deprecated and will be removed in 25.3. 

To remove this warning, please choose a default channel explicitly with conda's regular configuration system, e.g. by adding 'defaults' to the list of channels:

  conda config --add channels defaults

For more information see https://docs.conda.io/projects/conda/en/stable/user-guide/configuration/use-condarc.

In [2]:
# python imports 
import polars as pl
import plotly.io as pio
# VS Code draws the plotly mimetype natively (fast); the HTML copy keeps charts in the slides export
pio.renderers.default = "plotly_mimetype+notebook_connected"

In [3]:
# Read eBird csv data
ebird = pl.read_csv("data/ebird.csv") 
observations = pl.read_csv("data/observation.csv") 
species = pl.read_csv("data/899_Species.csv") 
header = pl.read_csv(
    "data/header.csv",
    null_values=["NA", ""],
    infer_schema_length=100_000,
    schema_overrides={"k_Latitude": pl.Float64, "k_Longitude": pl.Float64},
)
locations = pl.read_csv(
    "data/location.csv",
    null_values=["NA", ""],
    infer_schema_length=100_000,
    schema_overrides={"k_Latitude": pl.Float64, "k_Longitude": pl.Float64},
)

traits = pl.read_csv(
    "data/traits.csv",
    null_values=["NA", ""],
    infer_schema_length=100_000,
)
synonyms = pl.read_csv("data/synonyms.csv")


In [25]:
locations

bl_ID,bl_Location,bl_Lat,bl_Lon,region_Code,header_Count,Min_Alt,Max_Alt
i64,str,i64,i64,str,i64,f64,f64
2903,"""Aarons Pass""",3255,14945,"""T""",111,550.0,1000.0
2904,"""Abbeywood""",3125,14835,"""WS""",137,200.0,300.0
2905,"""Abercombie""",3355,14915,"""WS""",23,350.0,1000.0
2906,"""Abercrombie Caves""",3355,14925,"""WS""",35,400.0,1100.0
2907,"""Aberfoyle Sugarloaf""",3015,15205,"""NTE""",43,450.0,1350.0
…,…,…,…,…,…,…,…
6258,null,3755,14955,null,1,null,null
6259,null,3755,15445,null,1,null,null
6260,null,3755,15545,null,1,null,null


In [5]:
%%capture
# Rename variable names in eBird 
# In polars, we can apply a dictionary to rename columns
ebird = ebird.rename({"K-Latitude" : "key_latitude", "K-Longitude": "key_longitude"})
ebird = ebird.rename({"Latitude" : "latitude", "Longitude": "longitude"})

In [6]:
# Helper function that converts form degree:minutes to degree:decimal
def ddmm_to_dd(column_name: str) -> pl.Expr:
    degree = pl.col(column_name) // 100
    minutes = pl.col(column_name) - degree * 100
    return degree + (minutes / 60)

ebird = ebird.with_columns(
    key_latitude = -ddmm_to_dd("key_latitude"), 
    key_longitude = ddmm_to_dd("key_longitude")
)

ebird = ebird.with_columns(
    latitude = -ddmm_to_dd("latitude"), 
    longitude = ddmm_to_dd("longitude")
)




In [8]:
# Change ebird start and end dates from string to date type (format is YYYY/MM/DD)
ebird = ebird.with_columns(
    pl.col("Observation Start Date", "Observation End Date").str.to_date("%Y/%m/%d")
)

## Slide 1 - Sydney Heatmap
The following graph is a heatmap of bird occurences in sydney...

In [9]:
import plotly.figure_factory as ff

# Approximate Greater Sydney extent (Blue Mountains to coast, Central Coast to Wollondilly)
SYD_LAT = (-34.35, -32.99)
SYD_LON = (149.97, 151.63)

sydney_ebird = ebird.filter(
    pl.col("key_latitude").is_between(*SYD_LAT),
    pl.col("key_longitude").is_between(*SYD_LON),
)

fig = ff.create_hexbin_mapbox(
    data_frame=sydney_ebird, lat="key_latitude", lon="key_longitude",
    nx_hexagon=10, opacity=0.5, labels={"color": "Point Count"},
    mapbox_style="open-street-map"
)

fig.update_layout(margin = dict(b = 0, t = 0, l = 0, r = 0)) 
fig.show()


The hotspots are wollemi national park, hornsby and wollogong.

## eBird Observation Histogram

In [10]:
# Code for observation histogram
import plotly.express as px

# Count eBird observations per year, using the observation start date
obs_per_year = (
    ebird
    .group_by(pl.col("Observation Start Date").dt.year().alias("Year"))
    .len(name="Observations")
    .sort("Year")
)

In [11]:
# Frequency table of eBird observations per year, in a scrollable box
from IPython.display import HTML

freq_table = (
    ebird
    .group_by(pl.col("Observation Start Date").dt.year().alias("Year"))
    .len(name="Observations")
    .sort("Year")
    .with_columns(
        (pl.col("Observations") / pl.col("Observations").sum() * 100).round(2).alias("Percent")
    )
)

# Show every row, then wrap the table in a box that scrolls once it is 400px tall
with pl.Config(tbl_rows=-1):
    table_html = freq_table._repr_html_()

HTML(f'<div style="max-height: 400px; overflow-y: auto;">{table_html}</div>')

Year,Observations,Percent
i32,u32,f64
1951,5,0.0
1952,293,0.02
1953,265,0.01
1954,269,0.01
1955,212,0.01
1956,221,0.01
1957,186,0.01
1958,64,0.0
1962,13,0.0


- The *earliest* observation: 27th August 1951 
- The *latest* observation start is 31st March 2020 

- Dates are strings

## eBird heatmap

In [12]:
# Code for eBird Heatmap
import plotly.express as px

# NSW extent (same bounds as the R notebook)
NSW_LAT = (-37.7, -28.0)
NSW_LON = (140.5, 164.0)

# eBird key coordinates are the centres of 10-minute Bird Atlas grid cells,
# so each tile reaches 5 minutes either side of its centre
HALF_CELL = 5 / 60

# Count observations in each grid cell
nsw_cells = (
    ebird
    .filter(
        pl.col("key_latitude").is_between(*NSW_LAT),
        pl.col("key_longitude").is_between(*NSW_LON),
    )
    .group_by("key_latitude", "key_longitude")
    .len(name="Observations")
    .with_row_index("cell_id")
    .with_columns(log_observations=pl.col("Observations").log10())
)

# Turn each grid cell into a square polygon so plotly can draw it as a tile
def cell_square(lat, lon):
    return [[
        [lon - HALF_CELL, lat - HALF_CELL],
        [lon + HALF_CELL, lat - HALF_CELL],
        [lon + HALF_CELL, lat + HALF_CELL],
        [lon - HALF_CELL, lat + HALF_CELL],
        [lon - HALF_CELL, lat - HALF_CELL],
    ]]

grid_geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "id": row["cell_id"],
            "geometry": {"type": "Polygon", "coordinates": cell_square(row["key_latitude"], row["key_longitude"])},
        }
        for row in nsw_cells.iter_rows(named=True)
    ],
}

# Plasma runs yellow -> orange -> pink -> purple -> deep blue, changing hue as well as
# lightness so neighbouring values are easy to tell apart. It is also colourblind-safe.
# Reversed so yellow = few observations and deep blue = many.
COLOUR_SCALE = px.colors.sequential.Plasma_r

# Counts range from 1 to over 100,000, so colour by log10 and label the scale in real counts
max_log = nsw_cells["log_observations"].max()
tick_values = list(range(0, int(max_log) + 1))
tick_labels = [f"{10 ** v:,}" for v in tick_values]

fig = px.choropleth_mapbox(
    nsw_cells,
    geojson=grid_geojson,
    locations="cell_id",
    color="log_observations",
    color_continuous_scale=COLOUR_SCALE,
    range_color=(0, max_log),
    custom_data=["Observations", "key_latitude", "key_longitude"],
    mapbox_style="open-street-map",
    center={"lat": -32.8, "lon": 147.0},
    zoom=4.6,
    opacity=0.85,
    title="Heatmap of eBird observations in NSW",
)
fig.update_traces(
    marker_line_width=0,
    hovertemplate="%{customdata[0]:,} observations<br>Cell centre: %{customdata[1]:.2f}, %{customdata[2]:.2f}<extra></extra>",
)
fig.update_layout(
    height=650,
    margin=dict(l=0, r=0, t=50, b=0),
    coloraxis_colorbar=dict(title="Observations", tickvals=tick_values, ticktext=tick_labels),
)
fig.show()

## DES Notes 
- Number of observations in the DES Data  
- As this data is more recent, compare reporting rates to pre-2020 surveys 
- Are there any previously reported species missing in the DES data. 
- Where are the DES reports concentrated. 
- Isolate and analyze interesting / popular species 
- The des data has region codes
- It's an xslx file but the first table has everything we need to know about each survey. 
- 



### Regions 
- There are 9 regions in the dataset identified by a code.  
- What are the significance of these regions?  
- Rename the regions from codes to their actual names. 

In [14]:
# Reading DES data: only the first sheet of the spreadsheet
# engine="openpyxl" because polars' default Excel reader (fastexcel) isn't installed
des = pl.read_excel(
    "data/des.xlsx",
    sheet_name="DES_Backlog_Format",
    engine="openpyxl",
)
des

Survey ID,DES Ref,Latitude,Longitude,Observation Start Date,Observation End Date,Locality,Comments,Collection flag,Observer Id,Observer Last Name,Observer First Name,Location name,Region code,Species Common Name,Species ID,Breeding Code,Quality flag,Species Comments [Notes],K-Latitude,K-Longitude
i64,str,i64,i64,datetime[μs],datetime[μs],str,str,str,str,str,str,str,str,str,str,str,null,str,str,str
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Emu""","""1""","""Y""",null,"""NULL""","""2945""","""15035"""
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Masked Lapwing""","""133""","""Y""",null,"""NULL""","""2945""","""15035"""
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Australian Wood Duck""","""202""",null,null,"""NULL""","""2945""","""15035"""
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Pacific Black Duck""","""208""",null,null,"""NULL""","""2945""","""15035"""
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Australasian Grebe""","""61""",null,null,"""NULL""","""2945""","""15035"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
902577,"""BPC18-26044.033""",3155,14925,2022-08-21 00:00:00,2022-08-21 00:00:00,"""Moggymill""","""NULL""","""NULL""","""26044""","""Goulder""","""Graham""","""Moggymill""","""WS""","""Stubble Quail""","""9""",null,null,"""NULL""","""3155""","""14925"""
902577,"""BPC18-26044.033""",3155,14925,2022-08-21 00:00:00,2022-08-21 00:00:00,"""Moggymill""","""NULL""","""NULL""","""26044""","""Goulder""","""Graham""","""Moggymill""","""WS""","""Australian Magpie""","""705""",null,null,"""NULL""","""3155""","""14925"""
902577,"""BPC18-26044.033""",3155,14925,2022-08-21 00:00:00,2022-08-21 00:00:00,"""Moggymill""","""NULL""","""NULL""","""26044""","""Goulder""","""Graham""","""Moggymill""","""WS""","""Willie Wagtail""","""364""",null,null,"""NULL""","""3155""","""14925"""


In [15]:
# Converts DES coordinates from degrees-minutes (DDMM) to decimal degrees
# The K- columns are a cleaned copy of Latitude/Longitude: they fix small typos
# (e.g. 15034 -> 15035) and hold the text "NULL" where the original was unusable.
des = des.rename({
    "K-Latitude": "key_latitude", "K-Longitude": "key_longitude",
    "Latitude": "latitude", "Longitude": "longitude",
})

# Turn the text "NULL" into a real missing value, then make the key columns numbers
des = des.with_columns(
    pl.col("key_latitude", "key_longitude").replace("NULL", None).cast(pl.Int64)
)

# A DDMM value is only a real coordinate if its minutes are below 60 and its degrees
# are inside Australia's range. Anything else (e.g. 30015 or 15275) becomes missing.
def valid_ddmm(column_name: str, min_degree: int, max_degree: int) -> pl.Expr:
    degree = pl.col(column_name) // 100
    minutes = pl.col(column_name) % 100
    return (minutes < 60) & degree.is_between(min_degree, max_degree)

des = des.with_columns(
    pl.when(valid_ddmm(c, 9, 45)).then(pl.col(c)).otherwise(None).alias(c)
    for c in ["latitude", "key_latitude"]
).with_columns(
    pl.when(valid_ddmm(c, 112, 160)).then(pl.col(c)).otherwise(None).alias(c)
    for c in ["longitude", "key_longitude"]
)

# Same conversion as for eBird: latitude is south, so it becomes negative
des = des.with_columns(
    key_latitude=-ddmm_to_dd("key_latitude"),
    key_longitude=ddmm_to_dd("key_longitude"),
    latitude=-ddmm_to_dd("latitude"),
    longitude=ddmm_to_dd("longitude"),
)

des.select("latitude", "longitude", "key_latitude", "key_longitude").describe()

statistic,latitude,longitude,key_latitude,key_longitude
str,f64,f64,f64,f64
"""count""",104103.0,104125.0,104103.0,104125.0
"""null_count""",37.0,15.0,37.0,15.0
"""mean""",-31.436353,151.045275,-31.436357,151.045286
"""std""",1.804477,2.290407,1.804481,2.290405
"""min""",-37.416667,141.083333,-37.416667,141.083333
"""25%""",-33.25,149.916667,-33.25,149.916667
"""50%""",-31.083333,151.416667,-31.083333,151.416667
"""75%""",-30.083333,152.916667,-30.083333,152.916667
"""max""",-21.083333,153.583333,-21.083333,153.583333


In [16]:
# Number of regions in the dataset - 9 Regions  

des_sighting_locations = des['Region code'].value_counts(sort = True)  


region_names = {"NC": "Northern Coast", 
                "RP": "Riverine Plane", 
                "A": "Arid", 
                "CSC": "Central and South Coast", 
                "WS": "Western Slopes", 
                "T": "Tablelands", 
                "MAW": "Malee and Acacia Woodlands", 
                "CP": "Central Peneplain",
                "TS": "Tasman Sea"
                }

In [19]:
# DES heatmap: number of observations in each 10-minute grid cell across NSW

# Keep only locations inside NSW (NSW_LAT and NSW_LON are defined in the eBird heatmap cell)
des_nsw = des.filter(
    pl.col("key_latitude").is_between(*NSW_LAT),
    pl.col("key_longitude").is_between(*NSW_LON),
)

# Frequency table of locations, with an id for each grid cell
location_counts = (
    des_nsw
    .select("key_latitude", "key_longitude")
    .drop_nulls()
    .group_by("key_latitude", "key_longitude")
    .len("observations")
    # len() gives an unsigned integer, which plotly mistakes for categories
    # (one layer per distinct count). Casting to Int64 keeps it numeric.
    .with_columns(pl.col("observations").cast(pl.Int64))
    .with_row_index("cell_id")
)

# Put each cell into a logarithmic band. left_closed=True puts exactly 10 in "10–99", not "1–9".
BAND_LABELS = ["1–9", "10–99", "100–999", "1,000–9,999", "10,000+"]
location_counts = location_counts.with_columns(
    band=pl.col("observations").cut(
        breaks=[10, 100, 1000, 10000], labels=BAND_LABELS, left_closed=True
    )
)

# Each key coordinate is the centre of a 10-minute square, so h is half its width
h = 5 / 60

def square(lat, lon):
    return [[[lon - h, lat - h], [lon + h, lat - h],
             [lon + h, lat + h], [lon - h, lat + h], [lon - h, lat - h]]]

shapes = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature", "id": row["cell_id"],
         "geometry": {"type": "Polygon", "coordinates": square(row["key_latitude"], row["key_longitude"])}}
        for row in location_counts.iter_rows(named=True)
    ],
}

band_colours = dict(zip(BAND_LABELS, px.colors.sample_colorscale("Plasma_r", 5)))

fig = px.choropleth_mapbox(
    location_counts, geojson=shapes, locations="cell_id", color="band",
    color_discrete_map=band_colours,
    category_orders={"band": BAND_LABELS},   # legend runs from fewest to most
    custom_data=["observations"],
    mapbox_style="open-street-map", center={"lat": -32.8, "lon": 147.0}, zoom=4.6,
    opacity=0.85,
    title="Heatmap of DES observations in NSW: 2020 - 2026",
)
fig.update_traces(marker_line_width=0, hovertemplate="%{customdata[0]:,} observations<extra></extra>")
fig.update_layout(height=650, margin=dict(l=0, r=0, t=50, b=0), legend_title_text="Observations")
fig.show()

## Observations heatmap
The observations table has no coordinates, so each sighting takes its location from the survey it belongs to (`sd_ID` in observations matches `s_ID` in the survey headers).

In [20]:
# Observations heatmap: number of sightings in each 10-minute grid cell across NSW

# Attach each sighting to its survey to get the survey's grid-cell coordinates.
# An inner join drops the ~1,200 sightings whose survey isn't in the header table.
obs_with_location = (
    observations
    .join(header.select("s_ID", "k_Latitude", "k_Longitude"), left_on="sd_ID", right_on="s_ID")
    .rename({"k_Latitude": "key_latitude", "k_Longitude": "key_longitude"})
    # Header coordinates are DDMM like eBird's, and latitude is south
    .with_columns(
        key_latitude=-ddmm_to_dd("key_latitude"),
        key_longitude=ddmm_to_dd("key_longitude"),
    )
)

# Count sightings per grid cell inside NSW (a handful of typo'd cells fall outside and drop out here)
obs_counts = (
    obs_with_location
    .filter(
        pl.col("key_latitude").is_between(*NSW_LAT),
        pl.col("key_longitude").is_between(*NSW_LON),
    )
    .group_by("key_latitude", "key_longitude")
    .len("observations")
    .with_columns(pl.col("observations").cast(pl.Int64))   # keep plotly treating it as numeric
    .with_row_index("cell_id")
)

# Busiest cells pass 100,000 sightings, so this needs two more bands than the DES map
OBS_BAND_LABELS = ["1–9", "10–99", "100–999", "1,000–9,999", "10,000–99,999", "100,000+"]
obs_counts = obs_counts.with_columns(
    band=pl.col("observations").cut(
        breaks=[10, 100, 1000, 10000, 100000], labels=OBS_BAND_LABELS, left_closed=True
    )
)

# Grid-cell squares, using square() from the DES heatmap cell
obs_shapes = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature", "id": row["cell_id"],
         "geometry": {"type": "Polygon", "coordinates": square(row["key_latitude"], row["key_longitude"])}}
        for row in obs_counts.iter_rows(named=True)
    ],
}

obs_band_colours = dict(zip(OBS_BAND_LABELS, px.colors.sample_colorscale("Plasma_r", len(OBS_BAND_LABELS))))

fig = px.choropleth_mapbox(
    obs_counts, geojson=obs_shapes, locations="cell_id", color="band",
    color_discrete_map=obs_band_colours,
    category_orders={"band": OBS_BAND_LABELS},
    custom_data=["observations"],
    mapbox_style="open-street-map", center={"lat": -32.8, "lon": 147.0}, zoom=4.6,
    opacity=0.85,
    title="Heatmap of observations in NSW from 1960 - 2019",
)
fig.update_traces(marker_line_width=0, hovertemplate="%{customdata[0]:,} observations<extra></extra>")
fig.update_layout(height=650, margin=dict(l=0, r=0, t=50, b=0), legend_title_text="Observations")
fig.show()

# Ragav's Research questions  

- Spatial range-shift type maps. Such as where the Rock Warbler was found in 1960 against where it was found in 2010. 

    - One graph that shows all spottings of Rock warlblers in 1960 juxtaposed with where it was found in the 2010. 

    - What has changed in the last 50 years? 

- Are bird species consistenly found in one region or are they found in multiple regions?  

- Enumerate the number of unique bird species in one region for each decade from 1960 - 2010.

- Closely Observe patterns of endangered bird species. 
    -  Obtain a list of endangered bird species from the given dataset. 
    - Trends 
        - Is the reporting rate of these bird species declining? 
        - Number of habitats declining? 
        - How varied are the diets. 
    - If something showing an increasing or declining pattern. Is there any co-variate responsible for this? 

- Classify species into two groups: Declining species and Increasing species. 
    - Were any "increasing" species previosly present in the "declining" group and vice versa? 
    - Is there anything trait (Habitat, diet, region) common about declining species?    
        - Do further research on why that particular trait is in endangered? 

- Association between rainfall patterns / habitat destruction. 
    - Habitat destructions in NSW. 
    - Rainfall patterns in NSW
        - Look at the rainfall across trends in NSW.
            - Dry regions
            - Low rainfall 
            - Moderate rainfall
            - High rainfall
        






